In [ ]:
# Cell 0: Imports
import random, time, math, os, json as json_mod
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from itertools import product
from torch.utils.data import DataLoader
import networkx as nx
import pandas as pd

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 1: Config & Constants
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

_USE_AMP = (device == "cuda")

# Plot style
METHOD_COLORS = {"dense": "#4C72B0", "window": "#DD8452", "graph": "#55A868"}
METHOD_LABELS = {"dense": "Dense", "window": "Window", "graph": "Graph-Masked"}

plt.rcParams.update({
    "figure.facecolor":    "white",
    "axes.facecolor":      "#FAFAFA",
    "axes.edgecolor":      "#CCCCCC",
    "axes.grid":           True,
    "grid.color":          "#E0E0E0",
    "grid.linewidth":      0.6,
    "font.size":           11,
    "axes.titlesize":      13,
    "axes.titleweight":    "bold",
    "axes.labelsize":      11,
    "legend.fontsize":     10,
    "legend.framealpha":   0.9,
    "legend.edgecolor":    "#CCCCCC",
    "lines.linewidth":     2.0,
    "lines.markersize":    6,
    "xtick.direction":     "out",
    "ytick.direction":     "out",
    "figure.dpi":          120,
    "savefig.dpi":         150,
    "savefig.bbox":        "tight",
})

In [ ]:
# Cell 2: Data Pipeline
def build_dataset(n=4000, num_nodes=30, k=3, distractors=40, seed=None, KMAX=8):
    if seed is not None:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    base = num_nodes
    vocab = {"U": base, "V": base+1, "E": base+2, "SEP": base+3, "S": base+4}
    for i in range(KMAX+1):
        vocab[f"K{i}"] = base + 5 + i
    V = base + 5 + (KMAX+1)

    nodes = list(range(num_nodes))
    data = []

    def _try_one_sample():
        path = random.sample(nodes, k+1)
        true = [(path[i], path[i+1]) for i in range(k)]
        start, target = path[0], path[-1]
        true_set = set(true)
        cand = [(u, v) for u in nodes for v in nodes
                if u != v and (u, v) not in true_set]
        if distractors > len(cand):
            return None
        random.shuffle(cand)
        dist_edges = cand[:distractors]
        G = nx.DiGraph()
        G.add_edges_from(true + dist_edges)
        for p in nx.all_simple_paths(G, start, target, cutoff=k):
            if len(p) == k + 1 and p != path:
                return None
        edges = true + dist_edges
        return (edges, start, target, k)

    max_attempts = n * 3
    attempts = 0
    while len(data) < n and attempts < max_attempts:
        attempts += 1
        sample = _try_one_sample()
        if sample is not None:
            data.append(sample)

    if len(data) < n:
        print(f"Warning: only generated {len(data)}/{n} samples after {attempts} attempts")

    return data, V, vocab


def encode_sample(edges, start, target, k, vocab):
    blocks = [[vocab["U"], u, vocab["V"], v, vocab["E"]] for (u, v) in edges]
    random.shuffle(blocks)  # edge-order augmentation
    x = [t for b in blocks for t in b] + [vocab["SEP"], vocab["S"], start, vocab[f"K{k}"]]
    return torch.tensor(x, dtype=torch.long)


_DATASET_CACHE = {}

def clear_dataset_cache():
    _DATASET_CACHE.clear()

def get_dataset(n, num_nodes, k, distractors, seed):
    key = (n, num_nodes, k, distractors, seed)
    if key not in _DATASET_CACHE:
        _DATASET_CACHE[key] = build_dataset(n=n, num_nodes=num_nodes, k=k, distractors=distractors, seed=seed)
    return _DATASET_CACHE[key]


def split_data(data, val_frac=0.2):
    data = list(data)
    random.shuffle(data)
    n = len(data)
    n_val = int(n * val_frac)
    return data[n_val:], data[:n_val]


def collate_augmented(batch, vocab):
    xs = torch.stack([encode_sample(*sample, vocab) for sample in batch])
    ys = torch.tensor([sample[2] for sample in batch], dtype=torch.long)
    return xs, ys


def collate_fixed(batch):
    xs = torch.stack([b[0] for b in batch], dim=0)
    ys = torch.tensor([b[1] for b in batch], dtype=torch.long)
    return xs, ys

print("Data pipeline ready.")

In [ ]:
# Cell 3: Mask Functions
def sliding_window_mask(L, window, device):
    idx = torch.arange(L, device=device)
    return (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() > window


_GRAPH_MASK_CACHE = {}

def clear_graph_mask_cache():
    _GRAPH_MASK_CACHE.clear()

def graph_token_mask(x, vocab, num_nodes, device):
    B, L = x.shape
    sep_id = vocab["SEP"]
    sep_pos = (x[0] == sep_id).nonzero(as_tuple=False)
    if len(sep_pos) == 0:
        return torch.zeros(B, L, L, dtype=torch.bool, device=device)
    q_start = int(sep_pos[0].item())

    key = (L, q_start, num_nodes, str(device))
    cached = _GRAPH_MASK_CACHE.get(key)

    if cached is None:
        block = 5
        nb = q_start // block
        base = torch.arange(nb, device=device) * block
        node_pos = torch.cat([base + 1, base + 3], dim=0)
        A = torch.zeros(L, L, dtype=torch.bool, device=device)
        bid = torch.arange(q_start, device=device) // block
        A[:q_start, :q_start] = (bid[:, None] == bid[None, :])
        A[q_start:, q_start:] = True
        A[q_start:, node_pos] = True
        A[node_pos, q_start:] = True
        _GRAPH_MASK_CACHE[key] = (A, node_pos)
    else:
        A, node_pos = cached

    Ab = A.unsqueeze(0).expand(B, -1, -1).clone()
    vals = x[:, node_pos]
    valid = (vals >= 0) & (vals < num_nodes)
    eq = (vals[:, :, None] == vals[:, None, :]) & valid[:, :, None] & valid[:, None, :]
    Ab[:, node_pos.unsqueeze(1), node_pos.unsqueeze(0)] |= eq
    return ~Ab  # True=blocked


def khop_graph_token_mask(x, vocab, num_nodes, device, k_hops=2):
    B, L = x.shape
    sep_id = vocab["SEP"]
    sep_pos = (x[0] == sep_id).nonzero(as_tuple=False)
    if len(sep_pos) == 0:
        return torch.zeros(B, L, L, dtype=torch.bool, device=device)
    q_start = int(sep_pos[0].item())
    block = 5
    nb_edges = q_start // block

    A_base = torch.zeros(L, L, dtype=torch.bool, device=device)
    bid = torch.arange(q_start, device=device) // block
    A_base[:q_start, :q_start] = (bid[:, None] == bid[None, :])
    A_base[q_start:, q_start:] = True
    base_pos = torch.arange(nb_edges, device=device) * block
    node_pos = torch.cat([base_pos + 1, base_pos + 3], dim=0)
    A_base[q_start:, node_pos] = True
    A_base[node_pos, q_start:] = True

    Ab = A_base.unsqueeze(0).expand(B, -1, -1).clone()
    vals = x[:, node_pos]
    valid = (vals >= 0) & (vals < num_nodes)

    for b in range(B):
        node_ids = vals[b]
        v = valid[b]
        adj = torch.zeros(num_nodes, num_nodes, dtype=torch.float32, device=device)
        for ei in range(nb_edges):
            src_id = x[b, ei * block + 1].item()
            dst_id = x[b, ei * block + 3].item()
            if 0 <= src_id < num_nodes and 0 <= dst_id < num_nodes:
                adj[src_id, dst_id] = 1.0
                adj[dst_id, src_id] = 1.0

        reach = adj.clone()
        power = adj.clone()
        for _ in range(k_hops - 1):
            power = power @ adj
            reach = reach + power
        reach = (reach > 0)
        reach.fill_diagonal_(True)

        for i, ni in enumerate(node_pos):
            if not v[i]: continue
            nid_i = node_ids[i].item()
            for j, nj in enumerate(node_pos):
                if not v[j]: continue
                nid_j = node_ids[j].item()
                if reach[nid_i, nid_j]:
                    Ab[b, ni, nj] = True

    return ~Ab


def expand_mask_for_heads(m, h):
    B, L, _ = m.shape
    return m.unsqueeze(1).expand(B, h, L, L).reshape(B * h, L, L)

print("Mask functions ready.")

In [ ]:
# Cell 4: Model
class Block(torch.nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = torch.nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4*d_model),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(4*d_model, d_model),
        )
        self.drop = torch.nn.Dropout(dropout)
        self.ln1 = torch.nn.LayerNorm(d_model)
        self.ln2 = torch.nn.LayerNorm(d_model)

    def forward(self, h, attn_mask=None):
        a, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        h = self.ln1(h + self.drop(a))
        h = self.ln2(h + self.drop(self.ff(h)))
        return h


class TinyModel(torch.nn.Module):
    def __init__(self, V, num_nodes, sep_id, d_model=128, n_heads=4, n_layers=2, max_len=512, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.sep_id = sep_id
        self.tok = torch.nn.Embedding(V, d_model)
        self.pos = torch.nn.Embedding(max_len, d_model)
        self.drop = torch.nn.Dropout(dropout)
        self.blocks = torch.nn.ModuleList([Block(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])
        self.head = torch.nn.Linear(d_model, num_nodes)

    def forward(self, x, attn_mask=None):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        h = self.drop(self.tok(x) + self.pos(pos))
        for blk in self.blocks:
            h = blk(h, attn_mask=attn_mask)
        sep_pos = (x == self.sep_id).int().argmax(dim=1)
        sep_h = h[torch.arange(B, device=x.device), sep_pos]
        return self.head(sep_h)

print("Model ready.")

In [ ]:
# Cell 5: Training (AMP, GradScaler, warmup+cosine LR, label smoothing)

@torch.no_grad()
def eval_acc(model, loader, method, vocab=None, num_nodes=None, window=32,
             max_batches=None, cached_win=None, graph_mask_fn=None):
    _mask_fn = graph_mask_fn or graph_token_mask
    model.eval()
    x0, _ = next(iter(loader)); L = x0.shape[1]
    win = cached_win if cached_win is not None else (
        sliding_window_mask(L, window, device=device) if method == "window" else None)
    c = t = 0
    for bi, (x, y) in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        if method == "dense":
            attn = None
        elif method == "window":
            attn = win
        elif method == "graph":
            attn = expand_mask_for_heads(
                _mask_fn(x, vocab, num_nodes, device=device), model.n_heads)
        else:
            raise ValueError(method)
        with torch.amp.autocast('cuda', enabled=_USE_AMP):
            p = model(x, attn_mask=attn).argmax(-1)
        c += (p == y).sum().item()
        t += y.numel()
    return c / t if t else float("nan")


def train_one(method, num_nodes, vocab, V, train_loader, val_loader,
              n_layers=2, d_model=128, n_heads=4, dropout=0.1,
              window=32, epochs=20, lr=1e-3,
              eval_every=1, eval_val_batches=None,
              patience=5, min_delta=1e-4, mask_stats_batches=4,
              label_smoothing=0.05, warmup_epochs=3, graph_mask_fn=None):

    _mask_fn = graph_mask_fn or graph_token_mask
    model = TinyModel(V, num_nodes, sep_id=vocab["SEP"],
                      d_model=d_model, n_heads=n_heads,
                      n_layers=n_layers, dropout=dropout).to(device)

    if device == "cuda" and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
        except Exception:
            pass

    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler('cuda', enabled=_USE_AMP)

    def lr_lambda(ep):
        if ep < warmup_epochs:
            return (ep + 1) / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    x0, _ = next(iter(train_loader)); L = x0.shape[1]
    win = sliding_window_mask(L, window, device=device) if method == "window" else None

    best = 0.0; bad = 0; hist = []
    gsum = 0.0; gn = 0

    for ep in range(1, epochs + 1):
        t0 = time.time(); model.train()
        esum = en = 0.0

        for bi, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            if method == "dense":
                attn = None
            elif method == "window":
                attn = win
            elif method == "graph":
                m = _mask_fn(x, vocab, num_nodes, device=device)
                if bi < mask_stats_batches:
                    esum += (~m).float().mean().item(); en += 1
                attn = expand_mask_for_heads(m, model.n_heads)
            else:
                raise ValueError(method)

            with torch.amp.autocast('cuda', enabled=_USE_AMP):
                loss = F.cross_entropy(model(x, attn_mask=attn), y,
                                       label_smoothing=label_smoothing)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

        scheduler.step()

        if method == "graph" and en:
            gsum += esum; gn += en

        if (ep % eval_every) == 0 or ep == epochs:
            tr = eval_acc(model, train_loader, method, vocab=vocab, num_nodes=num_nodes,
                          window=window, max_batches=10, cached_win=win, graph_mask_fn=_mask_fn)
            va = eval_acc(model, val_loader, method, vocab=vocab, num_nodes=num_nodes,
                          window=window, max_batches=eval_val_batches, cached_win=win,
                          graph_mask_fn=_mask_fn)
            hist.append((ep, tr, va))

            if va > best + min_delta:
                best, bad = va, 0
            else:
                bad += 1

            cur_lr = scheduler.get_last_lr()[0]
            print(f"  {method} ep{ep}: train={tr:.3f} | val={va:.3f} | best={best:.3f} | "
                  f"bad={bad}/{patience} | lr={cur_lr:.5f} | {time.time()-t0:.1f}s")

            if bad >= patience:
                print(f"  {method} early stop @ ep{ep} (best={best:.3f})")
                break

    if method == "dense":
        allowed_pct = 100.0
    elif method == "window":
        allowed_pct = 100 * (~win).float().mean().item()
    else:
        allowed_pct = 100 * (gsum / max(gn, 1))
    print(f"  {method} allowed ~{allowed_pct:.1f}%")

    final_full = eval_acc(model, val_loader, method, vocab=vocab, num_nodes=num_nodes,
                          window=window, max_batches=None, cached_win=win, graph_mask_fn=_mask_fn)
    print(f"  {method} FINAL full-val acc = {final_full:.3f}")
    return model, best, final_full, allowed_pct, hist

print("Training functions ready.")

In [ ]:
# Cell 6: run_three
def run_three(k=3, distractors=40, num_nodes=30, n=4000, mask_hops=1,
              n_layers=2, d_model=128, n_heads=4, dropout=0.2,
              epochs=20, lr=1e-3,
              window=32, patience=7, seed=42,
              eval_every=1, eval_val_batches=None,
              fast=False, graph_mask_fn=None):
    clear_graph_mask_cache()

    if mask_hops > 1:
        def _khop_mask_fn(x, vocab, num_nodes, device, **kw):
            return khop_graph_token_mask(x, vocab, num_nodes, device, k_hops=mask_hops)
        graph_mask_fn = _khop_mask_fn

    if fast:
        epochs = min(epochs, 10)
        patience = min(patience, 3)
        eval_every = max(eval_every, 3)
        eval_val_batches = eval_val_batches or 5

    data, V, vocab = get_dataset(n, num_nodes, k, distractors, seed)

    random.seed(seed)
    train_raw, val_raw = split_data(data, 0.2)

    val_encoded = [(encode_sample(*s, vocab), s[2]) for s in val_raw]

    _pin = (device == "cuda")
    _vocab = vocab
    train_loader = DataLoader(train_raw, batch_size=64, shuffle=True,
                              collate_fn=lambda batch: collate_augmented(batch, _vocab),
                              pin_memory=_pin, num_workers=0)
    val_loader = DataLoader(val_encoded, batch_size=128, shuffle=False,
                            collate_fn=collate_fixed,
                            pin_memory=_pin, num_workers=2, persistent_workers=True)

    L = next(iter(train_loader))[0].shape[1]
    if n_layers is None:
        n_layers = 2

    mode_str = "FAST" if fast else "FULL"
    print(f"\nConfig [{mode_str}]: k={k}, d={distractors}, L={L}, V={V}, "
          f"n={len(data)}, layers={n_layers}, d_model={d_model}, "
          f"heads={n_heads}, dropout={dropout}, epochs={epochs}, patience={patience}")

    results = {"_L": L}
    for method in ("dense", "window", "graph"):
        print(f"\n-- {method.upper()} --")
        model, best_va, final_full, allowed_pct, hist = train_one(
            method, num_nodes, vocab, V, train_loader, val_loader,
            n_layers=n_layers, d_model=d_model, n_heads=n_heads, dropout=dropout,
            window=window, epochs=epochs, lr=lr,
            eval_every=eval_every, eval_val_batches=eval_val_batches,
            patience=patience, graph_mask_fn=graph_mask_fn
        )
        results[method] = {
            "best": best_va, "final_full": final_full,
            "allowed_pct": allowed_pct, "hist": hist
        }
    return results

print("run_three ready.")

In [ ]:
# Cell 7: Plotting Helpers
def plot_hist(results, title_suffix=""):
    fig, ax = plt.subplots(figsize=(10, 6))
    for m in ("dense", "window", "graph"):
        h = results[m]["hist"]
        if not h: continue
        e  = [t[0] for t in h]
        tr = [t[1] for t in h]
        va = [t[2] for t in h]
        c = METHOD_COLORS[m]
        ax.plot(e, va, marker="o", color=c, label=f"{METHOD_LABELS[m]} (val)", markersize=5)
        ax.plot(e, tr, marker=".", linestyle="--", color=c, alpha=0.45,
                label=f"{METHOD_LABELS[m]} (train)")
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_title(f"Training & Validation Accuracy{title_suffix}", fontsize=14, fontweight='bold')
    ax.legend(loc="upper left", ncol=2, fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_best_so_far(results, title_suffix=""):
    fig, ax = plt.subplots(figsize=(10, 6))
    for m in ("dense", "window", "graph"):
        h = results[m]["hist"]
        if not h: continue
        e  = [t[0] for t in h]
        va = [t[2] for t in h]
        best = []
        cur = -1.0
        for a in va:
            cur = max(cur, a)
            best.append(cur)
        ax.plot(e, best, marker="o", color=METHOD_COLORS[m],
                label=METHOD_LABELS[m], markersize=5)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Best Val Accuracy", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_title(f"Convergence (Best So Far){title_suffix}", fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

print("Plotting helpers ready.")

In [ ]:
# Cell 8: EXPERIMENT - k=1 Multi-Seed (THE MAIN EXPERIMENT)
# Expected runtime: ~45 min on T4
print("=" * 60)
print("MAIN EXPERIMENT: k=1 Multi-Seed")
print("=" * 60)

k1_results = {}
for d in [20, 40, 80]:
    k1_results[d] = {"dense": [], "window": [], "graph": []}
    for seed in range(5):
        r = run_three(k=1, distractors=d, n=20000, epochs=50,
                      patience=15, seed=seed, eval_every=1, eval_val_batches=None)
        for m in ["dense", "window", "graph"]:
            k1_results[d][m].append(r[m]["best"])
        print(f"  d={d}, seed={seed}: dense={r['dense']['best']:.1%}, "
              f"window={r['window']['best']:.1%}, graph={r['graph']['best']:.1%}")
    print(f"  d={d} AVG: dense={sum(k1_results[d]['dense'])/5:.1%}, "
          f"window={sum(k1_results[d]['window'])/5:.1%}, "
          f"graph={sum(k1_results[d]['graph'])/5:.1%}\n")

# Save last run's full results for convergence plotting
k1_last_run = r

In [ ]:
# Cell 9: EXPERIMENT - k=2 Multi-Seed
# Expected runtime: ~30 min on T4
print("=" * 60)
print("EXPERIMENT: k=2 Multi-Seed")
print("=" * 60)

k2_results = {}
for d in [20, 40, 80]:
    k2_results[d] = {"dense": [], "window": [], "graph": []}
    for seed in range(3):
        r = run_three(k=2, distractors=d, n=20000, epochs=50,
                      patience=15, seed=seed, eval_every=1, eval_val_batches=None)
        for m in ["dense", "window", "graph"]:
            k2_results[d][m].append(r[m]["best"])
        print(f"  d={d}, seed={seed}: dense={r['dense']['best']:.1%}, "
              f"window={r['window']['best']:.1%}, graph={r['graph']['best']:.1%}")
    print()

In [ ]:
# Cell 10: EXPERIMENT - k=3 Confirmation
print("=" * 60)
print("EXPERIMENT: k=3 Confirmation")
print("=" * 60)
r3 = run_three(k=3, distractors=40, n=20000, epochs=50,
               patience=15, seed=42, eval_every=1, eval_val_batches=None)
print(f"k=3: dense={r3['dense']['best']:.1%}, "
      f"window={r3['window']['best']:.1%}, graph={r3['graph']['best']:.1%}")

In [ ]:
# Cell 11: FIGURE 1 - Main Results Bar Chart (k=1)
fig, ax = plt.subplots(figsize=(10, 6))

ds = [20, 40, 80]
methods = ["dense", "window", "graph"]
x = np.arange(len(ds))
width = 0.22
random_baseline = 1.0 / 30

for i, m in enumerate(methods):
    means = [np.mean(k1_results[d][m]) for d in ds]
    stds  = [np.std(k1_results[d][m]) for d in ds]
    bars = ax.bar(x + i * width, means, width, yerr=stds, capsize=4,
                  color=METHOD_COLORS[m], label=METHOD_LABELS[m],
                  edgecolor='white', linewidth=0.5)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{mean:.0%}", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(y=random_baseline, color='gray', linestyle='--', alpha=0.7,
           label=f'Random ({random_baseline:.1%})')
ax.set_xlabel("Number of Distractor Edges", fontsize=12)
ax.set_ylabel("Validation Accuracy", fontsize=12)
ax.set_title("Graph-Masked Attention Accuracy (k=1)", fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([str(d) for d in ds])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=10)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("fig1_k1_accuracy.png", dpi=150)
plt.show()
print("Saved: fig1_k1_accuracy.png")

In [ ]:
# Cell 12: FIGURE 2 - Accuracy vs Hop Count
fig, ax = plt.subplots(figsize=(10, 6))

ks = [1, 2, 3]
for m in ["dense", "window", "graph"]:
    accs = []
    for k_val in ks:
        if k_val == 1:
            accs.append(np.mean(k1_results[40][m]))
        elif k_val == 2:
            accs.append(np.mean(k2_results[40][m]))
        else:
            accs.append(r3[m]["best"])
    ax.plot(ks, accs, marker='o', color=METHOD_COLORS[m], label=METHOD_LABELS[m],
            markersize=10, linewidth=2.5)

ax.axhline(y=1/30, color='gray', linestyle='--', alpha=0.7, label='Random (3.3%)')
ax.set_xlabel("Hop Count (k)", fontsize=12)
ax.set_ylabel("Best Validation Accuracy", fontsize=12)
ax.set_title("Accuracy Degrades with Hop Count (d=40)", fontsize=14, fontweight='bold')
ax.set_xticks(ks)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=10)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("fig2_accuracy_vs_hops.png", dpi=150)
plt.show()
print("Saved: fig2_accuracy_vs_hops.png")

In [ ]:
# Cell 13: FIGURE 3 - Convergence Curves (k=1, last run)
plot_hist(k1_last_run, title_suffix=" (k=1, d=80)")
plot_best_so_far(k1_last_run, title_suffix=" (k=1, d=80)")

In [ ]:
# Cell 14: FIGURE 4 - Attention Connectivity
fig, ax = plt.subplots(figsize=(10, 6))

methods = ["dense", "window", "graph"]
pcts = [k1_last_run[m]["allowed_pct"] for m in methods]
colors = [METHOD_COLORS[m] for m in methods]
labels = [METHOD_LABELS[m] for m in methods]

bars = ax.bar(labels, pcts, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{pct:.1f}%", ha='center', va='bottom', fontsize=13, fontweight='bold')

ax.set_ylabel("Attention Connections Used (%)", fontsize=12)
ax.set_title("Attention Sparsity by Mask Type", fontsize=14, fontweight='bold')
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig("fig4_sparsity.png", dpi=150)
plt.show()
print("Saved: fig4_sparsity.png")

In [ ]:
# Cell 15: FIGURE 5 - Sparsity vs Accuracy Scatter (The Sparsity Paradox)
fig, ax = plt.subplots(figsize=(10, 6))

for m in ["dense", "window", "graph"]:
    pct = k1_last_run[m]["allowed_pct"]
    acc = np.mean(k1_results[20][m])
    ax.scatter(pct, acc, color=METHOD_COLORS[m], s=200, zorder=5,
               edgecolors='black', linewidth=1.5)
    offset_x = 2 if m != "graph" else -2
    ha = 'left' if m != "graph" else 'right'
    ax.annotate(METHOD_LABELS[m], (pct, acc), fontsize=12, fontweight='bold',
                xytext=(offset_x, 10), textcoords='offset points', ha=ha)

ax.axhline(y=1/30, color='gray', linestyle='--', alpha=0.7, label='Random (3.3%)')
ax.set_xlabel("Attention Connections Used (%)", fontsize=12)
ax.set_ylabel("Validation Accuracy (k=1, d=20)", fontsize=12)
ax.set_title("The Sparsity Paradox: Less Attention = More Accuracy",
             fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("fig5_sparsity_paradox.png", dpi=150)
plt.show()
print("Saved: fig5_sparsity_paradox.png")

In [ ]:
# Cell 16: Summary Table
print("=" * 80)
print("SUMMARY OF RESULTS")
print("=" * 80)

print("\n--- k=1 Multi-Seed Results (5 seeds) ---")
print(f"{'Distractors':>12} {'Dense':>14} {'Window':>14} {'Graph':>14}")
print("-" * 58)
for d in [20, 40, 80]:
    dm = np.mean(k1_results[d]["dense"]); ds_ = np.std(k1_results[d]["dense"])
    wm = np.mean(k1_results[d]["window"]); ws = np.std(k1_results[d]["window"])
    gm = np.mean(k1_results[d]["graph"]); gs = np.std(k1_results[d]["graph"])
    print(f"{d:>12} {dm:>7.1%}+/-{ds_:.1%} {wm:>7.1%}+/-{ws:.1%} {gm:>7.1%}+/-{gs:.1%}")

print("\n--- k=2 Multi-Seed Results (3 seeds) ---")
print(f"{'Distractors':>12} {'Dense':>14} {'Window':>14} {'Graph':>14}")
print("-" * 58)
for d in [20, 40, 80]:
    dm = np.mean(k2_results[d]["dense"])
    wm = np.mean(k2_results[d]["window"])
    gm = np.mean(k2_results[d]["graph"])
    print(f"{d:>12} {dm:>13.1%} {wm:>13.1%} {gm:>13.1%}")

print("\n--- k=3 Single Run (seed=42, d=40) ---")
print(f"  Dense:  {r3['dense']['best']:.1%}")
print(f"  Window: {r3['window']['best']:.1%}")
print(f"  Graph:  {r3['graph']['best']:.1%}")

print(f"\nRandom baseline: {1/30:.1%}")
print("=" * 80)

In [ ]:
# Cell 17: Export Results
os.makedirs("results", exist_ok=True)

all_results = {
    "k1_multi_seed": {},
    "k2_multi_seed": {},
    "k3_single": {
        "dense": r3["dense"]["best"],
        "window": r3["window"]["best"],
        "graph": r3["graph"]["best"],
    }
}

for d in [20, 40, 80]:
    all_results["k1_multi_seed"][str(d)] = {
        m: {"values": k1_results[d][m],
            "mean": float(np.mean(k1_results[d][m])),
            "std": float(np.std(k1_results[d][m]))}
        for m in ["dense", "window", "graph"]
    }
    all_results["k2_multi_seed"][str(d)] = {
        m: {"values": k2_results[d][m],
            "mean": float(np.mean(k2_results[d][m])),
            "std": float(np.std(k2_results[d][m]))}
        for m in ["dense", "window", "graph"]
    }

with open("results/presentation_results.json", "w") as f:
    json_mod.dump(all_results, f, indent=2)
print("Saved: results/presentation_results.json")

rows = []
for k_val, res_dict, n_seeds in [(1, k1_results, 5), (2, k2_results, 3)]:
    for d in [20, 40, 80]:
        for m in ["dense", "window", "graph"]:
            rows.append({
                "k": k_val, "distractors": d, "method": m,
                "mean_acc": float(np.mean(res_dict[d][m])),
                "std_acc": float(np.std(res_dict[d][m])),
                "n_seeds": n_seeds,
                "values": str(res_dict[d][m]),
            })
for m in ["dense", "window", "graph"]:
    rows.append({
        "k": 3, "distractors": 40, "method": m,
        "mean_acc": r3[m]["best"], "std_acc": 0.0,
        "n_seeds": 1, "values": str([r3[m]["best"]]),
    })

df = pd.DataFrame(rows)
df.to_csv("results/presentation_results.csv", index=False)
print("Saved: results/presentation_results.csv")
print("\nDone! All results exported.")
print(df.to_string(index=False))